In [8]:
import os
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm  # Индикатор прогресса обучения

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import v2  # Обновленный модуль аугментаций PyTorch
!pip install -q scikit-learn 
import sklearn 
from sklearn.model_selection import (
    StratifiedKFold,
)  # Стратифицированное разбиение на фолды

# -----------------------------------------------------------------------------
# 1. Настройка окружения и воспроизводимости
# -----------------------------------------------------------------------------
SEED = 42  # Фиксированное число (зерно) для всех генераторов случайных чисел


def seed_everything(seed=42):
    random.seed(seed)  # Зашиваем случайность стандартной библиотеки Python
    os.environ["PYTHONHASHSEED"] = str(
        seed
    )  # Отключаем рандомизацию хэшей строк в Python
    np.random.seed(seed)  # Зашиваем случайность библиотеки NumPy
    torch.manual_seed(seed)  # Зашиваем случайность PyTorch для CPU
    torch.cuda.manual_seed(seed)  # Зашиваем случайность PyTorch для GPU
    torch.backends.cudnn.deterministic = (
        True  # Принудительно используем детерминированные алгоритмы cuDNN
    )


seed_everything(SEED)

# Автоматический выбор доступного вычислителя: Apple Silicon (mps), NVIDIA (cuda) или процессор (cpu)
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"Используем устройство: {device}")

# -----------------------------------------------------------------------------
# 2. Загрузка данных и подготовка
# -----------------------------------------------------------------------------
# Чтение CSV-файлов через pandas
train_df = pd.read_csv("data/digit_recognizer/train.csv")
test_df = pd.read_csv("data/digit_recognizer/test.csv")

# Извлечение массивов пикселей: drop отбрасывает колонку с правильным ответом "label"
X_raw = train_df.drop(columns=["label"]).values  # Размер: (42000, 784)
y_raw = train_df["label"].values  # Размер: (42000,)
X_test_raw = test_df.values  # Размер: (28000, 784)

# torch.tensor() — преобразует NumPy массив в PyTorch тензор
# dtype=torch.float32 — вычисления в сети происходят с плавающей точкой 32 бита
# .reshape(-1, 1, 28, 28) — меняем форму: -1 (авто-размер батча), 1 (канал), 28x28 (высота x ширина)
# / 255.0 — Мин-макс нормализация: переводим пиксели из [0, 255] в диапазон [0.0, 1.0]
X_train_all = (
    torch.tensor(X_raw, dtype=torch.float32).reshape(-1, 1, 28, 28) / 255.0
)
y_train_all = torch.tensor(
    y_raw, dtype=torch.long
)  # dtype=torch.long нужен для функции потерь CrossEntropy
X_test_all = (
    torch.tensor(X_test_raw, dtype=torch.float32).reshape(-1, 1, 28, 28)
    / 255.0
)

# -----------------------------------------------------------------------------
# 3. Кастомный Dataset и Аугментации
# -----------------------------------------------------------------------------
# v2.Compose — объединяет несколько последовательных трансформирующих операций
train_transforms = v2.Compose(
    [
        # degrees=10: поворот изображения случайным образом в диапазоне от -10 до +10 градусов
        v2.RandomRotation(degrees=10),
        # RandomAffine: геометрические искажения
        # degrees=0 (без поворота тут), translate=(0.08, 0.08) — сдвиг по X и Y до 8% от размера
        # scale=(0.92, 1.08) — случайное масштабирование картинки от 92% до 108%
        v2.RandomAffine(degrees=0, translate=(0.08, 0.08), scale=(0.92, 1.08)),
    ]
)


class DigitDataset(Dataset):

    def __init__(self, X, y=None, transform=None):
        self.X = X  # Тензор с изображениями
        self.y = y  # Тензор с метками классов
        self.transform = transform  # Аугментации

    def __len__(self):
        # Обязательный метод PyTorch Dataset: возвращает общее количество объектов
        return len(self.X)

    def __getitem__(self, idx):
        # Обязательный метод PyTorch Dataset: загрузка 1 объекта по его индексу idx
        x = self.X[idx]
        if self.transform:
            x = self.transform(
                x
            )  # Применяем аугментации к картинке, если они переданы
        if self.y is not None:
            return (
                x,
                self.y[idx],
            )  # Для train/val возвращаем и картинку, и метку
        return x  # Для test возвращаем только картинку


# -----------------------------------------------------------------------------
# 4. Профессиональная Архитектура CNN (ModernDigitCNN)
# -----------------------------------------------------------------------------
class ModernDigitCNN(nn.Module):

    def __init__(self, in_channels=1, num_classes=10):
        super().__init__()  # Инициализация базового класса nn.Module

        # Вспомогательный конструктор сверточного блока для уменьшения дублирования кода
        def conv_block(in_f, out_f):
            return nn.Sequential(
                # in_f/out_f: входы и выходы каналов; kernel_size=3: ядро 3x3
                # padding=1: сохраняет размерность картинки (28x28 остается 28x28)
                # bias=False: отключен, так как BatchNorm2d сам содержит сдвиг (beta)
                nn.Conv2d(in_f, out_f, kernel_size=3, padding=1, bias=False),
                # BatchNorm2d: нормализует активации по батчу, ускоряя сходимость
                nn.BatchNorm2d(out_f),
                # SiLU (Swish): активация x * sigmoid(x), работает лучше класического ReLU
                nn.SiLU(),
                nn.Conv2d(out_f, out_f, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_f),
                nn.SiLU(),
                # MaxPool2d(2, 2): уменьшает высоту и ширину карты в 2 раза (например с 28x28 до 14x14)
                nn.MaxPool2d(2, 2),
                # Dropout2d(0.15): с вероятностью 15% зануляет целиком карты признаков (каналы)
                nn.Dropout2d(0.15),
            )

        # Выделение признаков (Feature Extractor)
        self.features = nn.Sequential(
            conv_block(in_channels, 32),  # Блок 1: 1 -> 32 каналов (28x28 -> 14x14)
            conv_block(32, 64),  # Блок 2: 32 -> 64 каналов (14x14 -> 7x7)
            conv_block(64, 128),  # Блок 3: 64 -> 128 каналов (7x7 -> 3x3)
        )

        # Классификатор (Classifier Head)
        self.classifier = nn.Sequential(
            # AdaptiveAvgPool2d((1, 1)): превращает любую матрицу (например 128x3x3) в 128x1x1
            nn.AdaptiveAvgPool2d((1, 1)),
            # Flatten(): "расплющивает" тензор из (Batch, 128, 1, 1) в вектор (Batch, 128)
            nn.Flatten(),
            # Dropout(0.3): с вероятностью 30% обнуляет элементы вектора перед финальным слоем
            nn.Dropout(0.3),
            # Linear: полносвязный слой, переводит 128 признаков в 10 логитов (вероятностей для 10 цифр)
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        # Прямой проход: пропускаем x через свертки, а затем через классификатор
        return self.classifier(self.features(x))


# -----------------------------------------------------------------------------
# 5. Цикл обучения для 1 эпохи
# -----------------------------------------------------------------------------
def train_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()  # Переводим модель в режим обучения (активирует Dropout и BatchNorm)
    total_loss, correct = 0.0, 0

    for X, y in dataloader:
        X, y = X.to(device), y.to(
            device
        )  # Переносим батч данных на видеокарту/процессор

        logits = model(X)  # Прямой проход (forward pass)
        loss = loss_fn(logits, y)  # Вычисление ошибки

        optimizer.zero_grad()  # Обнуляем накопившиеся градиенты с прошлого шага
        loss.backward()  # Обратное распространение ошибки (backpropagation)
        optimizer.step()  # Обновление весов нейросети

        total_loss += loss.item() * len(
            y
        )  # Умножаем на размер батча для точного среднего
        preds = torch.argmax(
            logits, dim=1
        )  # Берем индекс наибольшего значения (предсказанный класс)
        correct += (
            (preds == y).sum().item()
        )  # Считаем количество верных совпадений

    # Возвращаем среднюю ошибку и точность (accuracy) за эпоху
    return total_loss / len(dataloader.dataset), correct / len(
        dataloader.dataset
    )


def eval_epoch(model, dataloader, loss_fn, device):
    model.eval()  # Переводим модель в режим оценки (выключает Dropout, BatchNorm фиксирует статистику)
    total_loss, correct = 0.0, 0

    # inference_mode(): выключает расчет градиентов (быстрее и экономит память, чем no_grad)
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            loss = loss_fn(logits, y)

            total_loss += loss.item() * len(y)
            preds = torch.argmax(logits, dim=1)
            correct += (preds == y).sum().item()

    return total_loss / len(dataloader.dataset), correct / len(
        dataloader.dataset
    )


# -----------------------------------------------------------------------------
# 6. Главный цикл K-Fold Cross-Validation
# -----------------------------------------------------------------------------
N_SPLITS = 5  # Разбиение на 5 фолдов
EPOCHS = 25  # Количество эпох обучения для каждого фолда
BATCH_SIZE = 64  # Размер пачки данных (батча)
LR = 1e-3  # Начальная скорость обучения (0.001)

# StratifiedKFold: делит выборку на 5 частей, сохраняя оригинальное процентное соотношение классов
# shuffle=True: перемешивание строк перед разбиением; random_state=SEED: фиксирует генератор
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Массив для хранения OOF (Out-Of-Fold) вероятностей размеров (42000, 10)
oof_predictions = np.zeros((len(X_train_all), 10))
test_models = []  # Список для сохранения 5 обученных моделей

print(f"=== Запуск {N_SPLITS}-Fold Кросс-Валидации ===")

# skf.split: возвращает индексы строк для train_idx и val_idx на каждом шаге
for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train_all, y_train_all)
):
    print(f"\n--- Fold {fold + 1}/{N_SPLITS} ---")

    # Формируем тренировочный датасет с аугментациями и валидационный без них
    train_sub = DigitDataset(
        X_train_all[train_idx],
        y_train_all[train_idx],
        transform=train_transforms,
    )
    val_sub = DigitDataset(
        X_train_all[val_idx], y_train_all[val_idx], transform=None
    )

    # DataLoader: разбивает датасет на батчи
    # shuffle=True: перемешивать батчи каждую эпоху; num_workers=0: обработка в основном потоке
    train_loader = DataLoader(
        train_sub, batch_size=BATCH_SIZE, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_sub, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0
    )

    # Инициализируем свежую модель для каждого фолда
    model = ModernDigitCNN().to(device)

    # AdamW: Adam с исправленной L2-регуляризацией (weight_decay=0.01)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LR, weight_decay=1e-2
    )

    # CosineAnnealingLR: планировщик, плавно понижающий learning rate по косинусоиде от LR до 0
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS
    )

    # CrossEntropyLoss с label_smoothing=0.1: сглаживает метки классов (вместо [0,1,0] делает [0.01, 0.91, 0.01])
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)

    best_val_acc = 0.0
    best_model_weights = None

    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(
            model, train_loader, optimizer, loss_fn, device
        )
        val_loss, val_acc = eval_epoch(model, val_loader, loss_fn, device)
        scheduler.step()  # Шаг изменения learning rate

        # Сохраняем веса лучшей эпохи (Checkpointing)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_weights = model.state_dict().copy()

        if (epoch + 1) % 5 == 0 or epoch == EPOCHS - 1:
            print(
                f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Val Acc: {val_acc:.4f}"
            )

    print(
        f"--> [Fold {fold + 1}] Лучшая Валидационная Точность: {best_val_acc:.5f}"
    )

    # Загружаем сохраненные лучшие веса эпохи обратно в модель
    model.load_state_dict(best_model_weights)
    test_models.append(model)

    # Получаем OOF вероятности для текущего фолда
    model.eval()
    val_probs = []
    with torch.inference_mode():
        for X_val, _ in val_loader:
            X_val = X_val.to(device)
            logits = model(X_val)
            # torch.softmax(..., dim=1): преобразует логиты в вероятности (сумма по строке = 1.0)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            val_probs.append(probs)

    # np.vstack: объединяет батчи вероятностей и записывает в оригинальные позиции фолда val_idx
    oof_predictions[val_idx] = np.vstack(val_probs)

# Расчет общего качества OOF по всей обучающей выборке (без утечки данных)
oof_acc = (oof_predictions.argmax(axis=1) == y_train_all.numpy()).mean()
print(f"\n==========================================")
print(f"ИТОГОВЫЙ OUT-OF-FOLD ACCURACY: {oof_acc:.5f}")
print(f"==========================================")

# -----------------------------------------------------------------------------
# 7. Инференс и Ансамблирование (Ensembling) по 5 Моделям
# -----------------------------------------------------------------------------
print("\nГенерация итоговых предсказаний (усреднение по 5 фолдам)...")

test_dataset = DigitDataset(X_test_all, transform=None)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False
)

final_test_probs = np.zeros((len(X_test_all), 10))

# Прогоняем тестовый набор через каждую из 5 сохраненных моделей
for fold_idx, model in enumerate(test_models):
    model.eval()
    fold_probs = []
    with torch.inference_mode():
        for X_test_batch in test_loader:
            X_test_batch = X_test_batch.to(device)
            logits = model(X_test_batch)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            fold_probs.append(probs)

    # Накапливаем вероятности, деля на количество моделей N_SPLITS (Soft Voting Ensemble)
    final_test_probs += np.vstack(fold_probs) / N_SPLITS

# Выбираем класс с максимальной усредненной вероятностью
final_preds = final_test_probs.argmax(axis=1)

# -----------------------------------------------------------------------------
# 8. Формирование корректного submission.csv
# -----------------------------------------------------------------------------
# Создаем итоговый DataFrame в формате Kaggle
submission = pd.DataFrame(
    {"ImageId": range(1, len(final_preds) + 1), "Label": final_preds}
)

# Выгружаем в CSV без индекса pandas
submission.to_csv("submission.csv", index=False)
print("Файл submission.csv успешно создан!")
submission.head(10)

Используем устройство: mps
=== Запуск 5-Fold Кросс-Валидации ===

--- Fold 1/5 ---
Epoch 05/25 | Train Loss: 0.5641 Acc: 0.9899 | Val Loss: 0.5390 Val Acc: 0.9925
Epoch 10/25 | Train Loss: 0.5441 Acc: 0.9940 | Val Loss: 0.5267 Val Acc: 0.9948
Epoch 15/25 | Train Loss: 0.5344 Acc: 0.9951 | Val Loss: 0.5197 Val Acc: 0.9954
Epoch 20/25 | Train Loss: 0.5277 Acc: 0.9971 | Val Loss: 0.5182 Val Acc: 0.9958
Epoch 25/25 | Train Loss: 0.5256 Acc: 0.9975 | Val Loss: 0.5170 Val Acc: 0.9961
--> [Fold 1] Лучшая Валидационная Точность: 0.99607

--- Fold 2/5 ---
Epoch 05/25 | Train Loss: 0.5646 Acc: 0.9893 | Val Loss: 0.5427 Val Acc: 0.9925
Epoch 10/25 | Train Loss: 0.5442 Acc: 0.9938 | Val Loss: 0.5298 Val Acc: 0.9940
Epoch 15/25 | Train Loss: 0.5350 Acc: 0.9949 | Val Loss: 0.5216 Val Acc: 0.9940
Epoch 20/25 | Train Loss: 0.5282 Acc: 0.9967 | Val Loss: 0.5170 Val Acc: 0.9961
Epoch 25/25 | Train Loss: 0.5257 Acc: 0.9971 | Val Loss: 0.5161 Val Acc: 0.9964
--> [Fold 2] Лучшая Валидационная Точность: 0.9

,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,0
4,5,3
5,6,7
6,7,0
7,8,3
8,9,0
9,10,3
